# Core Concepts: Wavefronts and Optical Elements

We'll explore the `Wavefront` object and how optical elements transform wavefronts through amplitude and phase modifications. We'll propagate wavefronts in different regimes and build a simple optical system.

In [ ]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt

A `Wavefront` stores the complex electric field on a grid, combining both amplitude and phase information. Let's start by setting up a computational grid and defining a circular pupil aperture — these are the basic building blocks we'll use throughout.

In [ ]:
# Create grid and aperture
pupil_diameter = 1
pupil_grid = make_pupil_grid(128, pupil_diameter * 1.2)
aperture = make_circular_aperture(1.0)(pupil_grid)

Now we'll build a planar wavefront from this aperture. Looking at the amplitude, phase, and intensity, we can see a perfect flat wavefront: the amplitude is uniform inside the pupil and zero outside, the phase is constant everywhere, and the intensity follows the same circular footprint as the amplitude.

In [ ]:
# Create wavefront and inspect properties
wavelength = 500e-9
wf = Wavefront(aperture, wavelength)

print(f"Wavelength: {wf.wavelength*1e9:.0f} nm")
print(f"Total power: {wf.total_power:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
imshow_field(wf.amplitude, ax=axes[0])
axes[0].set_title('Amplitude')
imshow_field(wf.phase, cmap='RdBu', ax=axes[1])
axes[1].set_title('Phase')
imshow_field(wf.intensity, ax=axes[2])
axes[2].set_title('Intensity')
plt.show()

Real optical systems introduce phase aberrations that distort the wavefront. We'll use Zernike polynomials to add two common aberrations: defocus and astigmatism. Defocus produces a parabolic phase map (bright center fading to dark edges), while astigmatism shows a distinctive saddle-shaped pattern with two orthogonal lobes.

In [ ]:
# Defocus
defocus_phase = zernike(2, 0)(pupil_grid)
wf_defocus = Wavefront(aperture * np.exp(1j * defocus_phase), wavelength)

# Astigmatism
astig_phase = zernike(2, 2)(pupil_grid)
wf_astig = Wavefront(aperture * np.exp(1j * astig_phase), wavelength)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imshow_field(wf_defocus.phase, cmap='RdBu', ax=axes[0])
axes[0].set_title('Defocus Phase')
imshow_field(wf_astig.phase, cmap='RdBu', ax=axes[1])
axes[1].set_title('Astigmatism Phase')
plt.show()

Next we'll propagate the planar wavefront to the focal plane using `FraunhoferPropagator`, which computes the far-field diffraction pattern — the point spread function (PSF). The resulting Airy pattern has a bright central core surrounded by concentric rings, with the first dark ring at a radius of about 1.22 λ/D from the center.

In [ ]:
wf_in = Wavefront(aperture, wavelength)

# Create focal grid and Fraunhofer propagator
focal_grid = make_focal_grid(q=4, num_airy=20, spatial_resolution=wavelength / pupil_diameter)

fraunhofer = FraunhoferPropagator(pupil_grid, focal_grid)
wf_focal = fraunhofer(wf_in)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imshow_field(wf_in.amplitude, ax=axes[0])
axes[0].set_title('Pupil Plane Amplitude')

# Log-scale PSF
img = wf_focal.intensity / wf_focal.intensity.max()
imshow_field(np.log10(img + 1e-20), cmap='inferno', vmin=-5, ax=axes[1])
axes[1].set_title('Focal Plane PSF (Airy Pattern)')
plt.show()

For shorter propagation distances, the Fraunhofer approximation breaks down, so we'll use `FresnelPropagator` instead. This accounts for the quadratic phase factor that accumulates during propagation. With a Fresnel number of about 200, we're firmly in the near-field regime where the intensity pattern still resembles the pupil shape with some edge diffraction effects.

In [ ]:
# Propagate with FresnelPropagator
distance = 10000.0  # meters
fresnel = FresnelPropagator(pupil_grid, distance)
wf_near = fresnel(wf_in)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
imshow_field(wf_near.intensity, cmap='inferno', ax=axes[0])
axes[0].set_title(f'Intensity at {distance}m')
imshow_field(wf_near.phase, cmap='RdBu', ax=axes[1])
axes[1].set_title(f'Phase at {distance}m')
plt.show()

print(f"Fresnel number: {pupil_diameter**2 / (wavelength * distance):.2f}")

Let's add some aberrations to see how wavefront errors degrade image quality. We'll create a phase screen — a sinusoidal ripple modulated by a Gaussian envelope — and apply it to the wavefront.

In [ ]:
# Create phase screen
phase_screen = 0.5 * np.sin(2 * np.pi * pupil_grid.x) * np.exp(-pupil_grid.as_('polar').r**2)

# Apply to wavefront
wf_aberrated = Wavefront(wf_in.electric_field * np.exp(1j * phase_screen), wavelength)

Now we propagate both the ideal and aberrated wavefronts to the focal plane and compare their PSFs. The Strehl ratio compares the peak focal-plane intensity of the aberrated PSF to the ideal one — values near 1 mean diffraction-limited performance, while our value will be noticeably lower, reflecting the phase errors we introduced.

In [ ]:
wf_focal_ideal = fraunhofer(wf_in)
wf_focal_aberrated = fraunhofer(wf_aberrated)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

imshow_field(phase_screen, cmap='RdBu', ax=axes[0,0])
axes[0,0].set_title('Phase Screen (radians)')

imshow_field(np.log10(wf_focal_ideal.intensity / wf_focal_ideal.intensity.max() + 1e-20), cmap='inferno', vmin=-5, ax=axes[0, 1])
axes[0,1].set_title('Ideal PSF')

imshow_field(wf_aberrated.phase, cmap='RdBu', ax=axes[1,0])
axes[1,0].set_title('Aberrated Wavefront Phase')

imshow_field(np.log10(wf_focal_aberrated.intensity / wf_focal_aberrated.intensity.max() + 1e-20), cmap='inferno', vmin=-5, ax=axes[1, 1])
axes[1,1].set_title('Aberrated PSF')

plt.tight_layout()
plt.show()

# Strehl ratio
strehl = wf_focal_aberrated.intensity.max() / wf_focal_ideal.intensity.max()
print(f"Strehl ratio: {strehl:.4f}")

HCIPy includes built-in optical element classes like `Apodizer`, which applies a transmission pattern to the wavefront amplitude. Here we take the aperture itself as the apodization mask and look at how the amplitude changes — the apodized wavefront's amplitude matches the original because we're essentially applying a circular mask to a wavefront that already has one.

In [ ]:
# Create Apodizer and apply to wavefront
aperture_field = make_circular_aperture(1.0)(pupil_grid)
apodizer = Apodizer(aperture_field)

wf_apodized = apodizer.forward(wf_in)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
imshow_field(wf_in.amplitude, ax=axes[0])
axes[0].set_title('Original Amplitude')
imshow_field(wf_apodized.amplitude, ax=axes[1])
axes[1].set_title('Apodized Amplitude')
plt.show()

To build a simple imaging system we need to add a lens. The lens imparts a quadratic phase curvature to the wavefront, which focuses the beam to a spot. We'll compute the lens phase from the focal length and wavelength.

In [ ]:
focal_length = 10.0  # meters

# Lens phase
lens_phase = -np.pi * (pupil_grid.x**2 + pupil_grid.y**2) / (wavelength * focal_length)

# Apply lens to wavefront
wf_after_lens = Wavefront(wf_in.electric_field * np.exp(1j * lens_phase), wavelength)

Now let's propagate the lens-modified wavefront to the focal plane and examine the result. The focused spot should be close to the diffraction-limited Airy disk size. Our system has an f-number of 10, so the Airy core radius is roughly 1.22 λ f/D, and we can check whether the focused spot matches this expectation.

In [ ]:
wf_focus = fraunhofer(wf_after_lens)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Lens phase pattern
imshow_field(lens_phase, cmap='RdBu', ax=axes[0])
axes[0].set_title('Lens Phase Pattern')

# Focal plane intensity
imshow_field(np.log10(wf_focus.intensity + 1e-20), vmin=-5, ax=axes[1])
axes[1].set_title('Focal Plane Intensity')

# Phase at focus
imshow_field(wf_focus.phase, cmap='RdBu', ax=axes[2])
axes[2].set_title('Phase at Focus')

plt.show()

print(f"Focal length: {focal_length} m")
print(f"f-number: {focal_length / pupil_diameter:.1f}")  # Diameter is 1.0 from aperture